In [1]:
# 変更部分を適用するリロードのおまじない
%load_ext autoreload
%autoreload 2

In [2]:
# ロガー準備
import logging
import sys

# 1. rootロガーを取得
logger = logging.getLogger()

# 2. ログレベルを設定 (DEBUG, INFO, WARNING など)
logger.setLevel(logging.INFO)

# 3. Jupyterの標準出力に書き出すハンドラを作成
# すでにハンドラが存在する場合は追加しない（二重出力を防ぐ）
if not logger.handlers:
    handler = logging.StreamHandler(sys.stdout)
    
    # 読みやすいフォーマットを設定
    formatter = logging.Formatter('%(levelname)s: %(name)s: %(message)s')
    handler.setFormatter(formatter)
    
    logger.addHandler(handler)

# テスト: これで engine.py 内のログも表示されるようになります
logging.info("Jupyterのロガー設定が完了しました。")

INFO: root: Jupyterのロガー設定が完了しました。


In [23]:
# 通常のCSVから読み込んだデータ
from src.simulator import RaceSimulator
from src.services.factory import CSVRaceFactory
from src.constants.enums import RaceEvent

sim = RaceSimulator(CSVRaceFactory())

date = "20260324"
course = "大井"
race_nums = [1]

info_list = sim.prepare(date=date, course=course, race_nums=race_nums)
#for h in info_list[0].profile.horses.values():
#    logger.info(h)
#for h in info_list[0].snapshot.horses.values():
#    logger.info(h)
sim.notify(RaceEvent.PREPARE, {'data': info_list})
race_info = info_list[0]

INFO: src.simulator: 初期化中...
INFO: src.core.engine: 初期化中...
INFO: src.services.saver: 初期化中...


INFO: src.services.factory: base_prof: {'race_id': '202603244401', 'course': '大井', 'race_name': 'C3五六', 'race_num': 1, 'num_horses': np.int64(13), 'distance': np.int64(1600), 'surface': 'ダ', 'condition': '良', 'weather': '晴'}
INFO: src.services.saver: prepared/202603244401_大井_ダ1600に結果を保存しました。


In [24]:
# レース実行
history = sim._run_single_race(info_list[0])

sim.notify(RaceEvent.FINISH, {'data': race_info, 'history': history})

INFO: src.services.saver: results/202603244401_大井_ダ1600に結果を保存しました。


In [ ]:
# デバッグテスト用
from src.simulator import RaceSimulator
from src.services.factory import DebugRaceFactory, DebugHorseFactory

sim = RaceSimulator(DebugRaceFactory())

info_list = [sim.factory.create_race()]
horses = {}
for i in range(0, 8):
    h_prof = sim.factory.horse_factory.create_horse_profile_as_random()
    horses[h_prof.horse_id] = h_prof
new_info = sim.factory.entry_horses(info_list[0], horses)
for h in new_info.profile.horses.values():
    logger.info(h)
for h in new_info.snapshot.horses.values():
    logger.info(h)

INFO: src.simulator: 初期化中...
INFO: src.core.engine: 初期化中...
INFO: root: HorseProfile(horse_id='D000000001', name='ダミー馬01', bracket_num=1, horse_num=2, jockey='ダミー機種01', horse_weight=514.6543883987812, weight_carried=56.0, max_speed=16.850225591977466, min_speed=14.47684908098593, acceleration=1.0232115038936442, total_stamina=2110.420478899461, stamina_waste_rate=0.9595513150058157, cornering_ability=0.4512152933019625, gate_reaction=0.7878876459887882, strategy='closer', target_spurt_dist=571.0122956855778)
INFO: root: HorseProfile(horse_id='D000000002', name='ダミー馬02', bracket_num=1, horse_num=3, jockey='ダミー機種02', horse_weight=364.4693272958506, weight_carried=55.0, max_speed=15.689049778308732, min_speed=14.097699932402149, acceleration=1.1740312806203388, total_stamina=2245.5433612616257, stamina_waste_rate=1.0138464361719823, cornering_ability=0.5887485173479173, gate_reaction=0.7129652819696144, strategy='rear', target_spurt_dist=665.3855574257371)
INFO: root: HorseProfile(horse_i